In [1]:
import pandas as pd
from pathlib import Path
import os
import plotly.express as px

os.chdir(Path.cwd().parent)

save_path= "backtest/attachments"

horizons = [1,3,5]
#horizons = [1]
strategies = {"momentum_score": "Momentum Strategie", "value_score": "Value Strategie"}

In [2]:
df = pd.read_csv("backtest/data/backtest_data.csv")
df["horizon"] = df["sellyear"] - df["buyyear"]
df = df[df["horizon"].isin(horizons)]

In [6]:
FONT = dict(family="Latin Modern Roman, Times New Roman, serif", size=14, color="#222222")
BLUE = "#4E79A7"      # Value
ORANGE = "#F28E2B"    # Momentum

# Mapping strategy-key -> Anzeigename -> Farbe automatisch aufbauen
color_map = {}
for strategy, s_name in strategies.items():
    if "value" in s_name.lower():
        color_map[strategy] = BLUE
    elif "momentum" in s_name.lower():
        color_map[strategy] = ORANGE
    else:
        # Fallback, falls Namen anders lauten: Reihenfolge im dict entscheidet
        color_map[strategy] = BLUE if len(color_map) == 0 else ORANGE

for h in horizons:
    df_filtered = df[df["horizon"] == h].sort_values("rendite").copy()

    sector_order = (
    df_filtered[df_filtered["strategy"] == "value_score"]
    .groupby("sector")["rendite"]
    .median()
    .sort_values(ascending=False)   # absteigend: bester Sektor zuerst
    .index
    .tolist()
    )

    fig = px.box(
        df_filtered,
        x="sector",
        y="rendite",
        color="strategy",
        color_discrete_map=color_map,
        points="outliers",   # Value & Momentum nebeneinander pro Sektor
        category_orders={"sector": sector_order},
    )

    fig.update_traces(
        marker=dict(size=2, opacity=0.7),
        line=dict(width=1.3),
        #width=0.3,   # macht die einzelnen Boxen schmaler
    )
    
    fig.update_xaxes(
        title="Sektor",
        showline=True,
        linecolor="#333333",
        ticks="outside",
        tickangle=-30,
        automargin=True,
    )
    fig.update_yaxes(
        title="Rendite p.a. (%)",
        showline=True,
        linecolor="#333333",
        ticks="outside",
        gridcolor="#e6e6e6",
        zeroline=True,
        zerolinecolor="#333333",
        zerolinewidth=1.2,
    )
    fig.update_layout(
        title=f"{h}-Jahres-Rendite nach Sektor — Value vs. Momentum",
        title_x=0.5,
        title_font=dict(size=18, family=FONT["family"]),
        font=FONT,
        width=1000,
        height=520,
        template="simple_white",
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(l=50, r=30, t=80, b=90),
        boxgap=0.2,        # Abstand zwischen den Box-Paaren je Sektor
        boxgroupgap=0.1,   # Abstand zwischen Value- und Momentum-Box innerhalb eines Sektors
        legend=dict(
            title="Strategie",
            orientation="h",
            yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
        ),
        showlegend=True,
    )
    fig.show()

    output_dir = Path(save_path) / "boxplot_comparison"
    output_dir.mkdir(parents=True, exist_ok=True)
    file_path = output_dir / f"boxplot_value_vs_momentum_{h}_year.png"
    fig.write_image(file_path, scale=3)

In [4]:
from pathlib import Path
import plotly.graph_objects as go

FONT = dict(family="Latin Modern Roman, Times New Roman, serif", size=14, color="#222222")
BLUE = "#4E79A7"      # Value
ORANGE = "#F28E2B"    # Momentum

for h in horizons:

    df_h = df[df["horizon"] == h].copy()

    fig = go.Figure()

    fig.add_trace(go.Violin(
        x=df_h["sector"][df_h["strategy"] == "value_score"],
        y=df_h["rendite"][df_h["strategy"] == "value_score"],
        legendgroup="Value", scalegroup="Value", name="Value",
        side="negative",
        line_color=BLUE,
        fillcolor=BLUE,
        opacity=0.6,
        points="outliers",             # dezente Ausreißerpunkte statt Box
        marker=dict(size=3, opacity=0.5),
        box_visible=False,
        meanline_visible=False,
        spanmode="hard"
    ))

    fig.add_trace(go.Violin(
        x=df_h["sector"][df_h["strategy"] == "momentum_score"],
        y=df_h["rendite"][df_h["strategy"] == "momentum_score"],
        legendgroup="Momentum", scalegroup="Momentum", name="Momentum",
        side="positive",
        line_color=ORANGE,
        fillcolor=ORANGE,
        opacity=0.6,
        points="outliers",
        marker=dict(size=3, opacity=0.5),
        box_visible=False,
        meanline_visible=False,
        spanmode="hard"
    ))

    fig.update_traces(line=dict(width=1.3))

    fig.update_xaxes(
        title="Sektor",
        showline=True, linecolor="#333333",
        ticks="outside", tickangle=-30, automargin=True,
        range=[-0.6, len(df_h["sector"].unique()) - 0.4],
    )

    fig.update_yaxes(
        title="Rendite p.a. (%)",
        showline=True, linecolor="#333333",
        ticks="outside", gridcolor="#e6e6e6",
        zeroline=True, zerolinecolor="#333333", zerolinewidth=1.2,
    )

    fig.update_layout(
        title=f"Renditeverteilung nach Sektor — {h}-Jahres-Horizont",
        title_x=0.5,
        title_font=dict(size=18, family=FONT["family"]),
        font=FONT,
        width=950, height=520,
        template="simple_white",
        plot_bgcolor="white", paper_bgcolor="white",
        margin=dict(l=30, r=30, t=80, b=90),
        violingap=0,
        violinmode="overlay",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5, title=None),
    )

    fig.show()

    output_dir = Path(save_path) / "violin_comparison"
    output_dir.mkdir(parents=True, exist_ok=True)
    fig.write_image(output_dir / f"violin_split_{h}_year.png", scale=3)

In [5]:
import pandas as pd

for h in horizons:

    df_h = df[df["horizon"] == h].copy()

    med = (
        df_h.groupby(["sector", "strategy"])["rendite"]
        .median()
        .unstack("strategy")
        .rename(columns={"value_score": "Value", "momentum_score": "Momentum"})
        .sort_values("Value")
    )

    display(med)

    fig = go.Figure()

    # Verbindungslinien
    for sector in med.index:
        fig.add_shape(
            type="line",
            x0=med.loc[sector, "Value"], x1=med.loc[sector, "Momentum"],
            y0=sector, y1=sector,
            line=dict(color="#999999", width=1.5),
            layer="below",
        )

    fig.add_trace(go.Scatter(
        x=med["Value"], y=med.index,
        mode="markers", name="Value",
        marker=dict(color=BLUE, size=11, line=dict(color="white", width=1)),
    ))

    fig.add_trace(go.Scatter(
        x=med["Momentum"], y=med.index,
        mode="markers", name="Momentum",
        marker=dict(color=ORANGE, size=11, line=dict(color="white", width=1)),
    ))

    fig.add_vline(x=0, line_width=1, line_color="#333333")

    fig.update_xaxes(
        title="Median-Rendite p.a. (%)",
        showline=True, linecolor="#333333",
        ticks="outside", gridcolor="#e6e6e6",
    )

    fig.update_yaxes(
        title="",
        showline=True, linecolor="#333333",
        ticks="outside", automargin=True,
    )

    fig.update_layout(
        title=f"Median-Rendite nach Sektor — {h}-Jahres-Horizont",
        title_x=0.5,
        title_font=dict(size=18, family=FONT["family"]),
        font=FONT,
        width=900, height=520,
        template="simple_white",
        plot_bgcolor="white", paper_bgcolor="white",
        margin=dict(l=50, r=30, t=80, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5, title=None),
    )

    fig.show()

    output_dir = Path(save_path) / "median_comparison"
    output_dir.mkdir(parents=True, exist_ok=True)
    fig.write_image(output_dir / f"median_dumbbell_{h}_year.png", scale=3)

strategy,Momentum,Value
sector,,
Real Estate,7.82,4.35
Communication Services,13.24,5.17
Consumer Defensive,10.62,8.21
Healthcare,16.08,13.34
Utilities,13.61,13.41
Energy,6.93,13.68
Financial Services,14.52,14.55
Consumer Cyclical,17.51,14.96
Industrials,15.73,16.81


strategy,Momentum,Value
sector,,
Consumer Cyclical,10.14,8.090
Communication Services,14.00,8.310
Energy,7.22,8.480
Technology,18.16,9.130
Industrials,14.02,9.490
Basic Materials,11.13,10.410
Real Estate,8.78,10.520
Utilities,12.34,10.705
Consumer Defensive,12.91,12.840


strategy,Momentum,Value
sector,,
Communication Services,14.07,5.88
Real Estate,8.03,8.05
Energy,6.53,9.31
Consumer Cyclical,13.59,9.34
Financial Services,12.59,9.82
Basic Materials,12.09,10.06
Consumer Defensive,9.46,10.28
Utilities,12.12,11.18
Industrials,14.86,13.72
